In [1]:
import os
import sys
import aim
import torch

os.environ["CONFIG_PATHS"] = "../configs/self_play_20.yaml"
os.environ["CONFIG_OVERRIDES"] = 'game.moves_directory="../data/moves_20_4"'
sys.path.append("../src")

from training.helpers import TrainingLoop
from neural_net import NeuralNet
from configuration import config

Loaded config:  {"development": {"check_move_validity": true, "profile": false, "runtime": 0, "display_logs_in_console": false, "output_directory": "data/2024-12-30_23-23-24-rubefaction", "create_initial_model": false}, "logging": {"save_interval": 600, "mcts_report_fraction": 0, "ucb_report": false, "gpu_evaluation": true, "made_move": true}, "game": {"board_size": 20, "num_moves": 30433, "num_pieces": 21, "num_piece_orientations": 91, "moves_directory": "../data/moves_20_4"}, "architecture": {"gameplay_processes": 6, "coroutines_per_process": 256, "game_flush_threshold": 100}, "training": {"run": true, "device": "mps", "network_name": "default", "batch_size": 128, "policy_loss_weight": 0.158, "learning_rate": 0.001, "sample_window": 50000, "samples_per_generation": 10000, "sampling_ratio": 2.0, "minimum_window_size": 10000, "new_data_check_interval": 60}, "networks": {"default": {"main_body_channels": 64, "residual_blocks": 10, "value_head_channels": 16, "value_head_flat_layer_width"

In [2]:
NETWORK_CONFIG = config()["networks"][config()["training"]["network_name"]]
LEARNING_RATE = config()["training"]["learning_rate"]

model = NeuralNet(NETWORK_CONFIG)
model.to("mps")
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

training_loop = TrainingLoop(
    initial_model=model,
    initial_lifetime_loaded_samples=0,
    optimizer=optimizer,
    device="mps",
    gamedata_dir="/Users/shivamsarodia/Dev/BlokusBot/data/2024-12-30_23-23-24-rubefaction/games",
    compute_top_one=True,
)

run = aim.Run(repo='/Users/shivamsarodia/Dev/BlokusBot/')
run["hparams"] = {
    "network": NETWORK_CONFIG,
    "config": config()["training"],
}
print("Starting training on run:", run.hash)

while True:
    action, result = training_loop.run_iteration()

    if action == "trained":
        for key, value in result.items():
            run.track(
                value,
                name=key,
                step=result["lifetime_trained_samples"],
            )

    if action == "no_new_data":
        print("All done!")
        break

run.close()

event | 1740122780.3507142 | training_initialized | {"lifetime_loaded_samples": 0, "lifetime_trained_samples": 0, "current_sampling_ratio": 100000, "target_sampling_ratio": 2.0, "window_size": 0}
Starting training on run: fa7584ee247e4c6f9c6a13fb
event | 1740122858.3569322 | training_dropped_from_window | {"num_dropped_samples": 397, "num_samples_remaining": 49648}
event | 1740122859.702601 | training_dropped_from_window | {"num_dropped_samples": 419, "num_samples_remaining": 49911}
event | 1740122861.103104 | training_dropped_from_window | {"num_dropped_samples": 409, "num_samples_remaining": 50176}
event | 1740122862.35678 | training_dropped_from_window | {"num_dropped_samples": 389, "num_samples_remaining": 50469}
event | 1740122863.7720199 | training_dropped_from_window | {"num_dropped_samples": 410, "num_samples_remaining": 50742}
event | 1740122865.114371 | training_dropped_from_window | {"num_dropped_samples": 430, "num_samples_remaining": 51029}
event | 1740122866.44736 | train

In [15]:
while True:
    action, result = training_loop.run_iteration()
    if action == "trained":
        print(action, result)
        break

trained {'value_loss': 0.010870804078876972, 'policy_loss': 0.005010528024286032, 'loss': 0.015881331637501717, 'lifetime_loaded_samples': 10328, 'lifetime_trained_samples': 128, 'current_sampling_ratio': 0.3902439024390244}


In [29]:
training_loop.run_iteration()

('trained',
 {'value_loss': 0.010762164369225502,
  'policy_loss': 0.005303236190229654,
  'loss': 0.01606540009379387,
  'lifetime_loaded_samples': 10988,
  'lifetime_trained_samples': 1792,
  'current_sampling_ratio': 1.8137651821862348})